In [1]:
import pandas as pd
import numpy as np

# --- Configuration (Adjust paths and row count) ---
# Set the path to your local files 
TD_TRAIN_PATH = 'train.csv'
TD_TEST_PATH = 'test_supplement.csv'
N_ROWS = 500_000 # Use a stable row limit; adjust up/down based on your RAM

# Define column types for extreme memory efficiency
dtypes = {
    'ip'            : 'uint32',
    'app'           : 'uint16',
    'device'        : 'uint16',
    'os'            : 'uint16',
    'channel'       : 'uint16',
    'click_time'    : 'object', # Keep as string until parsing
    'is_attributed' : 'uint8',
    'click_id'      : 'uint32'
}

# --- Load Data ---
print(f"Loading {N_ROWS} rows of train and all test data...")
try:
    td_train_df = pd.read_csv(
        TD_TRAIN_PATH,
        dtype=dtypes,
        usecols=['ip', 'app', 'device', 'os', 'channel', 'click_time', 'is_attributed'],
        nrows=N_ROWS
    )
    td_test_df = pd.read_csv(
        TD_TEST_PATH,
        dtype=dtypes,
        usecols=['ip', 'app', 'device', 'os', 'channel', 'click_time', 'click_id']
    )
except FileNotFoundError as e:
    print(f"FATAL ERROR: Could not load files. Please verify paths.")
    raise e

print(f"Train Shape: {td_train_df.shape}")
print(f"Test Shape: {td_test_df.shape}")

# --- Merging Train and Test ---
# Store test IDs for final submission
test_click_ids = td_test_df['click_id']
td_test_df.drop('click_id', axis=1, inplace=True)
td_test_df['is_attributed'] = -1 # Sentinel value to separate test set later

# Concatenate for global feature engineering
merged_df = pd.concat([td_train_df, td_test_df], ignore_index=True)

# Clean up memory immediately by deleting raw dataframes
del td_train_df
del td_test_df

print(f"Merged DataFrame shape: {merged_df.shape}")

Loading 500000 rows of train and all test data...
Train Shape: (500000, 7)
Test Shape: (57537505, 7)
Merged DataFrame shape: (58037505, 7)


In [2]:
print(merged_df.columns.tolist())

# Also check if the column is present
print('click_time' in merged_df.columns)

['ip', 'app', 'device', 'os', 'channel', 'click_time', 'is_attributed']
True


In [ ]:
import numpy as np
# Assumes merged_df exists from Section 1

# --- Utility Functions ---
def map_frequency(df, feature_col):
    """Calculates frequency using value_counts and maps it directly."""
    counts = df[feature_col].value_counts()
    freq_series = df[feature_col].map(counts)
    del counts # Delete the temporary counts Series
    return freq_series.astype('uint32')

def get_time_diff(df, groupby_cols, time_col, new_name):
    """Calculates time difference (next click - current click) using pandas rolling window."""
    # Sort before grouping
    df.sort_values(by=time_col, inplace=True)
    
    # Calculate the shift: the time of the next click for the same group
    df[new_name + '_next'] = df.groupby(groupby_cols)[time_col].shift(-1)
    
    # Calculate the time difference in seconds
    df[new_name] = (df[new_name + '_next'] - df[time_col]).dt.total_seconds().astype('float32')
    
    # Clean up and release memory
    df.drop([new_name + '_next'], axis=1, inplace=True)
    df.sort_index(inplace=True) # Restore original index order
    return df

# --- Feature Engineering Execution ---
print("--- Starting Feature Engineering and Merging ---")

# --- 0. Robust Time Parsing ---
merged_df['click_time'] = pd.to_datetime(merged_df['click_time'], errors='coerce')
merged_df['click_time'] = merged_df['click_time'].fillna(method='bfill').fillna(method='ffill')


# --- 1. TIME-BASED FEATURES ---
merged_df['day_of_week'] = merged_df['click_time'].dt.dayofweek.astype('uint8')
merged_df['hour_of_day'] = merged_df['click_time'].dt.hour.astype('uint8')
merged_df['minute_of_day'] = merged_df['click_time'].dt.minute.astype('uint8')

# --- 2. ADVANCED INTERACTION & TIME-DIFFERENCE FEATURES ---

# A. Time-to-Next-Click 
print("Calculating Time-to-Next-Click features...")
merged_df = get_time_diff(merged_df, ['ip'], 'click_time', 'next_click_ip')
merged_df = get_time_diff(merged_df, ['ip', 'device', 'os'], 'click_time', 'next_click_ip_dev_os')

# Drop the large timestamp column now
merged_df.drop('click_time', axis=1, inplace=True) 

# B. Highly Predictive Interaction Features (Created and immediately Frequency Encoded)
interaction_list = [
    (['ip', 'app', 'os'], 'ip_app_os_freq'), 
    (['ip', 'day_of_week', 'hour_of_day'], 'ip_day_hour_freq'),
    (['ip', 'device'], 'ip_device_freq'),
    (['app', 'channel'], 'app_channel_freq'),
]
print("Calculating Interaction Frequency features...")

for cols, new_name in interaction_list:
    # 1. Create a temporary string column
    temp_col = '_'.join([str(c) for c in cols])
    merged_df[temp_col] = merged_df[cols].astype(str).agg('_'.join, axis=1) 
    
    # 2. Map frequency directly to the new column
    merged_df[new_name] = map_frequency(merged_df, temp_col)
    
    # 3. Drop the temporary string column immediately
    merged_df.drop(temp_col, axis=1, inplace=True)


# --- 3. BASIC FREQUENCY ENCODING ---
freq_cols_basic = ['ip', 'app', 'device', 'os', 'channel']

print("Calculating Basic Frequency features...")
for col in freq_cols_basic:
    new_freq_name = col + '_freq'
    merged_df[new_freq_name] = map_frequency(merged_df, col)


# --- 4. FINAL CLEANUP and Typing ---
categorical_cols_final = ['ip', 'app', 'device', 'os', 'channel', 
                          'day_of_week', 'hour_of_day', 'minute_of_day']
for col in categorical_cols_final:
    if col in merged_df.columns:
        merged_df[col] = merged_df[col].astype('category')
        
print("--- Feature Engineering and Merging Complete ---")
print(f"Final Merged DataFrame shape: {merged_df.shape}")

--- Starting Feature Engineering and Merging ---


C:\Users\HP\AppData\Local\Temp\ipykernel_20836\1909689309.py:33: FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  merged_df['click_time'] = merged_df['click_time'].fillna(method='bfill').fillna(method='ffill')


Calculating Time-to-Next-Click features...
Calculating Interaction Frequency features...


In [ ]:
# Separate into final X and y
y = merged_df[merged_df['is_attributed'] != -1]['is_attributed']
X = merged_df[merged_df['is_attributed'] != -1].drop('is_attributed', axis=1)
X_submission = merged_df[merged_df['is_attributed'] == -1].drop('is_attributed', axis=1)

# Check Class Imbalance
imbalance_ratio = (y.sum() / len(y)) * 100

plt.figure(figsize=(7, 5))
sns.countplot(x=y, palette=['#1f77b4', '#ff7f0e'])
plt.title(f'Target Imbalance: is_attributed (Fraud/Download)', fontsize=14)
plt.xlabel('is_attributed (0: Not Downloaded, 1: Downloaded/Fraud)', fontsize=12)
plt.ylabel('Count (Log Scale)', fontsize=12)
plt.yscale('log')
plt.xticks([0, 1], ['Legitimate (0)', 'Fraud/Download (1)'])
plt.text(0.5, 100, f'Fraud/Download Rate: {imbalance_ratio:.4f}%', 
         horizontalalignment='center', color='red', fontsize=12, bbox=dict(facecolor='white', alpha=0.8))
plt.show()

print(f"\nPositive Class (1) Count: {y.sum()}")
print(f"Total Observations: {len(y)}")
print(f"Imbalance is severe: This confirms the necessity of Downsampling.")

In [ ]:
# Merge target back for plotting purposes
X_plot = X.copy()
X_plot['is_attributed'] = y

plt.figure(figsize=(12, 5))
sns.lineplot(
    data=X_plot.groupby('hour_of_day')['is_attributed'].mean().reset_index(),
    x='hour_of_day',
    y='is_attributed',
    marker='o',
    color='r'
)
plt.title('Fraud/Download Rate by Hour of Day 🕒', fontsize=16)
plt.xlabel('Hour of Day (0-23)', fontsize=12)
plt.ylabel('Attribution Rate (P(is_attributed=1))', fontsize=12)
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

NameError: name 'X' is not defined

In [ ]:
# Analyze the log distribution of the highly predictive 'ip_freq'
plt.figure(figsize=(10, 6))
sns.histplot(np.log1p(X['ip_freq']), bins=50, kde=True, color='purple')
plt.title('Log Distribution of IP Frequency Feature', fontsize=16)
plt.xlabel('log(1 + ip_freq)', fontsize=12)
plt.ylabel('Density', fontsize=12)
plt.show()

print("The heavily skewed distribution confirms that most IPs are rare, but a few are extremely common (a sign of bot activity).")

In [ ]:
# --- 1. Separate Feature Types ---
num_cols = [col for col in X.columns if '_freq' in col]
cat_cols = [col for col in X.columns if col not in num_cols]

# --- 2. Scaling Numerical Frequency Columns ---
scaler = StandardScaler()
X_scaled = X.copy()
X_scaled[num_cols] = scaler.fit_transform(X_scaled[num_cols])
print("Frequency features successfully scaled.")

# Apply scaling to the submission set as well
X_submission[num_cols] = scaler.transform(X_submission[num_cols])


# --- 3. Downsample (Handle Class Imbalance) ---
# We use RandomUnderSampler to reduce the majority class (is_attributed=0)
rus = RandomUnderSampler(sampling_strategy=0.5, random_state=42) # A common strategy is 1:2 ratio
X_bal, y_bal = rus.fit_resample(X_scaled, y)

print("\n--- Final Data Shapes ---")
print(f"Features (X) before Downsampling: {X_scaled.shape}")
print(f"Features (X_bal) after Downsampling: {X_bal.shape}")
print(f"Target (y_bal) after Downsampling: {y_bal.shape}")
print(f"Submission Data (X_submission) shape: {X_submission.shape}")
print(f"Balanced Positives in Training Set: {sum(y_bal)} (Ratio: {sum(y_bal)/len(y_bal):.4f})")

# The DataFrames X_bal and y_bal are now ready to be split into train/test sets
# and used for model training (e.g., LightGBM or XGBoost).